[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Causal_Inference.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Causal Inference

The question every regression dodges: what happens if we **intervene**? DAGs and d-separation, confounding and the backdoor adjustment, and the modern estimators — all on simulated worlds where we can *run the true intervention* and check the answer, the luxury real data never grants.

## 1. Pre-requisites

[Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb), [Independence](../Intro_Math/Analysis/Independence.ipynb); regression fluency ([Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
def ols(X, y):
    X1 = np.c_[np.ones(len(X)), X]
    return np.linalg.lstsq(X1, y, rcond=None)[0]

---
### 🕐 Session 1 of 3 — *Correlation, Confounding & the do-Operator* (~40 min)
**Goal:** watch a confounder manufacture a correlation; define intervention as graph surgery.
**Feeds into:** Session 2 (backdoor adjustment).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Correlation, Confounding & the do-Operator</b></summary>

**Timing (~40 min).** 10 min the question regression cannot answer · 12 min seeing versus doing · 10 min the simulation and its oracle · 8 min reading the damage.

**Open with the question, because it is the one every applied course dodges.** A regression tells you $E[Y \mid X = x]$ — what to expect of $Y$ among units *observed* at $X = x$. Almost nobody wants that. They want to know what happens **if we intervene**: change the policy, prescribe the drug, add the feature. Those are different quantities, and the entire workshop is about when and why they differ.

**Make the distinction physical before it is notational.** Observed $X$ **arrives with its causes attached**. If older people exercise less, then the group you observe exercising a lot is also the young group, and their better health has two sources. Setting $X$ by intervention severs that link — you, not the world, chose the value. Pearl's $do(\cdot)$ is exactly **graph surgery**: delete every arrow *into* $X$, leave the rest of the graph alone.

**Emphasise the methodological luxury this notebook has, since it is what makes the whole thing checkable.** We wrote the world, so we can *actually run the intervention* — call `world(n, do_x=5)` and the assignment line is overwritten. **Every estimate in this workshop is graded against a measured oracle, not against an assumed truth.** Real data never grants that, which is why causal claims in the wild are so contested, and why practising on simulated worlds is the right way to learn.

**Have the room predict the naive slope before running.** The true effect is 2.0 by construction. Most will guess the estimate comes out somewhat biased. It comes out at **5.4** — nearly triple. Let the size of the miss land, because "confounding inflates estimates a bit" is a much weaker intuition than "confounding can triple your answer with a sample size of 20,000."

**Then make the point that sample size is the wrong remedy, and say it in one sentence.** With $n = 20{,}000$ the standard error on that slope is around 0.02, so 5.4 is roughly **170 standard errors** away from the truth. **Bias does not shrink with $n$.** A confounded estimate is a precise measurement of the wrong quantity, and more data makes it more precise, not less wrong. This is the single most important sentence in the session.

**If the room is comfortable with algebra, decompose the 5.4 live.** $\mathrm{Cov}(X,Y) = 2\mathrm{Var}(X) - 0.5\mathrm{Cov}(X,Z)$, and since $X$ decreases in $Z$ while $Y$ also decreases in $Z$, the backdoor path contributes with the *same* sign as the causal effect. Dividing through gives $\approx 2 + 3.4$ — the causal part plus the laundered confounding. Seeing the two terms separately makes "the regression launders the backdoor path into the coefficient" concrete rather than metaphorical.

**Close by setting up the next session as a question with a non-obvious answer.** If adjusting for the confounder fixes this, why not adjust for everything you measured? Let the room reach the natural conclusion, and then promise that Session 2 will show it is wrong — that some variables *create* bias when you control for them. That cliffhanger is what makes the collider demo land.
</details>

## 2. Seeing vs Doing

💡 **Intuition.** $P(Y \mid X = x)$ answers 'what do I expect of Y among units *observed* to have X = x?' — but observed X carries its causes with it. $P(Y \mid do(X = x))$ answers 'what if I *set* X to x?' — **graph surgery**: delete the arrows into X, because your intervention, not the world, chose it. The two differ exactly when a **confounder** feeds both X and Y. Our simulated world makes this concrete because we can *actually perform* the do — rerun the world with X forced.

In [2]:
# World 1: exercise (X) → health (Y), both driven by age (Z). TRUE causal effect: +2.0
def world(n, do_x=None):
    Z = rng.uniform(20, 70, n)                                # age (confounder)
    X = np.clip(8 - 0.1*Z + rng.standard_normal(n), 0, None)  # older people exercise less
    if do_x is not None: X = np.full(n, float(do_x))          # THE INTERVENTION: cut Z→X
    Y = 2.0*X - 0.5*Z + 60 + 2*rng.standard_normal(n)         # health
    return Z, X, Y

Z, X, Y = world(20000)
naive_slope = ols(X[:, None], Y)[1]
# the interventional ORACLE: force X and measure the response directly
y_do = {x0: world(20000, do_x=x0)[2].mean() for x0 in (2.0, 5.0)}
true_effect = (y_do[5.0] - y_do[2.0]) / 3.0
print(f"naive regression slope of Y on X: {naive_slope:.2f}")
print(f"TRUE causal effect (measured by actually intervening): {true_effect:.2f}")
print("→ the naive slope is more than double the truth: age drives both low exercise and poor")
print("  health, and regression happily launders that path into the X coefficient")

naive regression slope of Y on X: 5.40
TRUE causal effect (measured by actually intervening): 1.98
→ the naive slope is more than double the truth: age drives both low exercise and poor
  health, and regression happily launders that path into the X coefficient


**What just happened.** The regression says the effect of exercise on health is **5.40**. The truth — measured by actually forcing $X$ and rerunning the world — is **1.98**, against the 2.0 written into the generating code. The regression is off by a factor of **2.7**.

**And it is not off because of noise.** With $n = 20{,}000$ the standard error on that slope is roughly 0.02, so 5.40 sits about **170 standard errors** from the truth. **This is bias, and bias does not shrink with sample size.** Collect ten million points and you get 5.40 with tighter error bars: a beautifully precise measurement of the wrong quantity. That single sentence is the most important thing in this session, and it is what separates a statistical problem from a causal one.

**Decompose the 5.40 and the mechanism is visible.** Since $Y = 2X - 0.5Z + \ldots$ and $X = 8 - 0.1Z + \varepsilon$:

$$\frac{\mathrm{Cov}(X,Y)}{\mathrm{Var}(X)} = \frac{2\mathrm{Var}(X) - 0.5\mathrm{Cov}(X,Z)}{\mathrm{Var}(X)} \approx \underbrace{2.0}_{\text{causal}} + \underbrace{3.4}_{\text{backdoor path via age}}$$

Age pushes exercise *down* and health *down*, so the two negatives multiply to a positive contribution — the confounding **adds to** the true effect rather than cancelling it. The regression cannot tell the two terms apart; it reports their sum and calls it "the coefficient on X".

**Which is why the intervention is a different computation, not a better one.** Look at the line `if do_x is not None: X = np.full(n, float(do_x))`. It **overwrites the assignment mechanism** — age no longer determines exercise, because we do. That is $do(\cdot)$ implemented literally: delete the arrows into $X$, leave everything else alone. The remaining $Z \to Y$ arrow still operates, which is why the intervened world still has age affecting health; we severed one edge, not the variable.

**Note what this demo has that no real study does.** We can run both worlds. In practice you observe one, and the interventional quantity is **not identified by the data alone** — it requires assumptions about the graph, which is why causal claims are argued rather than computed. Simulated worlds are the only place to *learn* these methods, precisely because they are the only place the answer key exists.

**Finally, the practical shape of the error.** A naive analyst here would report that a unit of exercise buys 5.4 health points, and would be confident about it. Acting on that number — funding a programme sized to a 5.4 effect — buys 2.0. **The failure is not that the model fits badly; it fits the observational data perfectly.** It answers a question nobody asked.

---
### 🕐 Session 2 of 3 — *d-Separation & the Backdoor Adjustment* (~40 min)
**Goal:** block the right paths: adjust for confounders, DON'T adjust for colliders — both verified.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (modern estimators).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: d-Separation & the Backdoor Adjustment</b></summary>

**Timing (~40 min).** 8 min backdoor adjustment · 15 min colliders · 10 min selection bias in the wild · 7 min d-separation as bookkeeping.

**Do the easy half first and quickly, because it confirms the room's intuition.** Adding age to the regression recovers 2.014 against a truth of 2.0. Conditioning on $Z$ **blocks** the backdoor path $X \leftarrow Z \rightarrow Y$, and what remains is the causal effect. Five minutes; everyone expected it.

**Then ask the question the success invites, and let them commit to an answer.** "So why not control for everything you measured?" Most rooms say yes, or say yes with a caveat about overfitting. Get the show of hands *before* running the collider cell — the demo is far more effective as a refutation of a belief they have just voiced than as a fact delivered cold.

**Teach the collider through its structure, not its name.** A collider is a variable that **two arrows point into**: $X \rightarrow C \leftarrow Y$. Unconditionally the path through it is **already blocked** — knowing your talent tells you nothing about your luck. Condition on $C$ and it **opens**: among admitted students, high talent implies the luck must have been low, because something had to get them in. That "something had to" is the entire mechanism, and the admissions story makes it land in one sentence.

**Give the numbers so it is a measurement rather than a metaphor.** $X$ and $Y$ are independent by construction, and the unadjusted slope is $-0.002$. Controlling for $C$ gives $-0.800$ — and that value is exactly predictable: solving the two-variable normal equations with $\mathrm{Var}(C) = 2.25$ and $\mathrm{Cov}(X,C) = \mathrm{Cov}(Y,C) = 1$ yields $-1/1.25 = -0.8$. **The bias is not an artifact of the sample; it is the population value.** Worth putting on the board if the room can take it — a bias you can predict analytically is much more convincing than one you merely observe.

**Then run the selection version, because it is where students will actually meet this.** Restricting to $C > 1$ gives $-0.490$: same phenomenon, no regression involved. **Selecting your sample on a collider is conditioning on it.** Name the real cases — studying only hospitalised patients, only admitted applicants, only employed workers, only surviving firms — and note that in each the analyst never typed the word "control". The bias arrives with the dataset.

**State the rule cleanly once the demos have earned it.** *Adjust for confounders; never adjust for colliders or their descendants.* And the corollary that overturns a common instinct: **"control for everything" is not conservative, it is a way to manufacture bias from thin air.** The graph — not the data, not a variable-selection algorithm — tells you which is which, because the two cases are statistically indistinguishable in the sample.

**Close on d-separation as the general bookkeeping.** Every path from $X$ to $Y$ is open or blocked according to three rules (chain, fork, collider), conditioning flips the collider rule, and $X$ is independent of $Y$ given $S$ exactly when every path is blocked. That is the complete algorithm behind everything in this session, and it is worth naming even if there is not time to develop it — students should know the rules they used are a special case of something systematic.
</details>

## 3. Which Variables to Control For

💡 **Intuition.** The graph answers it. Non-causal association flows along **backdoor paths** (X ← Z → Y); conditioning on Z *blocks* them — so regressing Y on X **and Z** recovers the causal slope. But the rule cuts both ways: a **collider** (X → C ← Y) is blocked *by default*, and conditioning on it **opens** a spurious path — 'controlling for everything' is how careful-sounding analyses create bias from thin air (selection on admission, hospitalization, employment...). d-separation is the complete bookkeeping of which paths are open.

In [3]:
# Backdoor adjustment on World 1: add the confounder to the regression
adj_slope = ols(np.c_[X, Z], Y)[1]
print(f"backdoor-adjusted slope (control for age): {adj_slope:.3f}   (truth 2.0 — recovered)")

# World 2: X and Y CAUSALLY UNRELATED, but both cause C (a collider)
def world2(n):
    X2 = rng.standard_normal(n)
    Y2 = rng.standard_normal(n)
    C2 = X2 + Y2 + 0.5*rng.standard_normal(n)              # e.g., 'got admitted' = talent + luck
    return X2, Y2, C2
X2, Y2, C2 = world2(20000)
print(f"\nWorld 2 — X, Y independent by construction:")
print(f"  slope of Y on X (correct: ~0):             {ols(X2[:, None], Y2)[1]:+.3f}")
print(f"  slope of Y on X, 'controlling' for C:      {ols(np.c_[X2, C2], Y2)[1]:+.3f}   ← collider bias, manufactured")
sel = C2 > 1.0
print(f"  slope among selected units (C > 1):        {ols(X2[sel][:, None], Y2[sel])[1]:+.3f}   ← same bias via selection")

backdoor-adjusted slope (control for age): 2.014   (truth 2.0 — recovered)

World 2 — X, Y independent by construction:
  slope of Y on X (correct: ~0):             -0.002
  slope of Y on X, 'controlling' for C:      -0.800   ← collider bias, manufactured
  slope among selected units (C > 1):        -0.490   ← same bias via selection


**What just happened.** Two results that point in opposite directions, and the contrast is the session:

| world | analysis | result | truth |
|---|---|---|---|
| 1 | regress $Y$ on $X$ | 5.40 | 2.0 |
| 1 | regress $Y$ on $X$ **and $Z$** | **2.014** | 2.0 |
| 2 | regress $Y$ on $X$ | −0.002 | 0 |
| 2 | regress $Y$ on $X$ **and $C$** | **−0.800** | 0 |
| 2 | regress on $X$ among $C > 1$ | **−0.490** | 0 |

**Adding a variable fixed the first world and broke the second.** Same action, opposite consequences — which is why "control for everything you measured" is not a conservative default but a way to manufacture bias from nothing.

**World 1 is the expected half.** Conditioning on age blocks the backdoor path $X \leftarrow Z \rightarrow Y$, leaving only the causal path, and 2.014 against a truth of 2.0 is within noise. Confounding removed.

**World 2 is the half that overturns the instinct, and the number is exactly predictable.** $X$ and $Y$ are drawn **independently**; there is no causal path between them at all. Yet controlling for $C = X + Y + 0.5\varepsilon$ produces $-0.800$, and that is not sampling noise — solving the normal equations with $\mathrm{Var}(C) = 2.25$ and $\mathrm{Cov}(X,C) = \mathrm{Cov}(Y,C) = 1$ gives $-1/1.25 = -0.8$ exactly. **The bias is the population value.** More data reproduces it more precisely.

**The mechanism, in one sentence you can say aloud.** $C$ is "got admitted" and $X + Y$ is "talent plus luck". Among people who got in, learning that someone has high talent tells you their luck must have been low — *something* had to get them in. **Conditioning on a common effect makes its independent causes dependent.** The negative sign is not incidental: it is the arithmetic of a fixed budget being split.

**The third row is where students will actually meet this, and it involves no regression at all.** Restricting to $C > 1$ gives $-0.490$. **Selecting your sample on a collider *is* conditioning on it.** Study only hospitalised patients, only admitted applicants, only employed workers, only firms that survived — and the bias arrives with the dataset, before any modelling decision is made. Nobody typed the word "control".

**Which yields the rule this session exists to establish.** Adjust for confounders; never adjust for colliders or their descendants. And note the uncomfortable part: **$Z$ and $C$ are statistically indistinguishable in the data.** Both correlate with $X$ and with $Y$. No variable-selection procedure, no significance test, no regularisation path can tell you which is which — only a claim about the *causal structure* can, and that claim comes from domain knowledge, not from the sample.

**Finally, note the general machinery these two cases are instances of.** d-separation says a path is blocked or open by three rules — chain, fork, collider — with conditioning *closing* the first two and *opening* the third. $X \perp Y \mid S$ exactly when every path between them is blocked. Both worlds above are one-line applications of that bookkeeping, and it scales to graphs far too large to reason about by story.

---
### 🕐 Session 3 of 3 — *Modern Estimators* (~40 min)
**Goal:** IPW, standardization, doubly-robust — three routes to the same do, audited against the oracle.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Modern Estimators</b></summary>

**Timing (~40 min).** 8 min the setup and the ATE · 10 min standardization · 10 min IPW · 7 min doubly robust · 5 min the honest boundary.

**Establish what is being estimated before any estimator appears.** The average treatment effect is $E[Y \mid do(T{=}1)] - E[Y \mid do(T{=}0)]$ — the difference between two *whole worlds*, one where everyone is treated and one where nobody is. Note that with a heterogeneous effect ($-0.5\,TZ$ in the code) this is an **average** over the population's $Z$ distribution, and $E[1.5 - 0.5Z] = 1.5$ since $E[Z] = 0$. Whose average matters: change the population and the ATE changes even though the mechanism does not.

**Give each estimator one sentence of intuition, and make them sound different, because they are.**
- *Standardization* fits $E[Y \mid T, Z]$ and then **averages the fitted model over everyone's $Z$** — simulating the two worlds using the outcome model.
- *IPW* reweights each unit by $1/P(T \mid Z)$, upweighting the units that were unlikely to receive the treatment they got. **It manufactures the randomised trial nature refused to run**, using only the treatment model.
- *Doubly robust* combines them and is consistent if **either** model is right.

**Emphasise the asymmetry of information, since it is the reason to have all three.** Standardization needs the outcome model correct and does not care about the propensity. IPW needs the propensity correct and does not care about the outcome. **They fail in different places**, which is exactly what makes combining them worthwhile — DR gives you two chances to be right instead of one. Frame it as insurance against your own misspecification, not as a smarter algorithm.

**Point at the propensity weights as the practical failure mode of IPW.** With $p = \sigma(1.5Z)$, a unit at $Z = 3$ has $p \approx 0.989$; an untreated unit there carries weight $1/(1-p) \approx 90$. **A handful of extreme weights can dominate the estimate.** In this simulation $Z$ is standard normal so the tails are thin and it behaves; with a stronger confounder or a heavier tail, IPW becomes wildly high-variance. Weight trimming and overlap diagnostics exist for exactly this, and it is worth naming even though the demo does not need them.

**Ask the room to predict the naive difference in means before running.** It comes out at 3.34 against a truth of 1.50 — more than double. The reason is worth decomposing: high-$Z$ units are both **more likely to be treated** and have **higher $Y$ regardless of treatment**, so the treated group is not comparable to the untreated group. Same confounding structure as Session 1, now with a binary treatment.

**Then read the results as an audit, which is what they are.** Oracle 1.502; standardization 1.490; IPW 1.498; DR 1.498. All three land within 0.012 of a **measured** oracle. Do not, however, let the room conclude that one is best — the spread is smaller than the Monte Carlo noise at $n = 40{,}000$, and this simulation has **both** models correctly specified. A demo where every model is right cannot demonstrate double robustness. If time allows, that is the experiment to suggest: deliberately misspecify one model and rerun.

**Close on the honest boundary, and give it real weight rather than treating it as a footnote.** Every method here assumed **no unmeasured confounding** — that $Z$ captures everything feeding both $T$ and $Y$. Nothing in the data can verify that assumption; a hidden confounder produces data that looks exactly the same. **The graph tells you what to adjust for; only the design tells you whether you measured enough.** That is why instrumental variables, sensitivity analysis, and randomised trials exist, and why a randomised trial is worth more than any estimator: randomisation *severs* the arrow into $T$ by construction rather than by assumption.
</details>

## 4. Three Roads to the Interventional Answer

💡 **Intuition.** With binary treatment, three standard estimators of $E[Y|do(T{=}1)] - E[Y|do(T{=}0)]$:
1. **Standardization**: model $E[Y|T,Z]$, average over the *whole* Z population.
2. **Inverse propensity weighting (IPW)**: reweight each unit by $1/P(T{=}t|Z)$ — manufacture the randomized trial that nature refused to run.
3. **Doubly robust**: combine both; consistent if *either* model is right — insurance against your own misspecification.
The oracle discipline continues: we simulate the true counterfactuals and grade all three.

In [4]:
# binary-treatment world: treatment probability depends on Z; effect heterogeneous
def world3(n, force=None):
    Z3 = rng.standard_normal(n)
    p_t = 1/(1 + np.exp(-1.5*Z3))                            # sicker (high Z) → more treated
    T = (rng.random(n) < p_t).astype(float) if force is None else np.full(n, float(force))
    Y3 = 1.5*T + 2.0*Z3 - 0.5*T*Z3 + rng.standard_normal(n)  # true ATE = E[1.5 − 0.5 Z] = 1.5
    return Z3, T, Y3

Z3, T, Y3 = world3(40000)
ate_oracle = world3(200000, force=1)[2].mean() - world3(200000, force=0)[2].mean()

naive = Y3[T==1].mean() - Y3[T==0].mean()

# 1) standardization (outcome regression with interaction)
b = ols(np.c_[T, Z3, T*Z3], Y3)
std_est = (b[0] + b[1]*1 + b[2]*Z3.mean() + b[3]*Z3.mean()) - (b[0] + b[2]*Z3.mean())

# 2) IPW with a logistic propensity (fit by Newton in 6 lines)
w_l = np.zeros(2)
Xl = np.c_[np.ones(len(Z3)), Z3]
for _ in range(30):                                          # Newton–Raphson for logistic regression
    p = 1/(1+np.exp(-Xl @ w_l))
    H = (Xl * (p*(1-p))[:, None]).T @ Xl
    w_l += np.linalg.solve(H, Xl.T @ (T - p))
p_hat = 1/(1+np.exp(-Xl @ w_l))
ipw_est = np.mean(T*Y3/p_hat) - np.mean((1-T)*Y3/(1-p_hat))

# 3) doubly robust (AIPW)
mu1 = b[0] + b[1] + (b[2]+b[3])*Z3
mu0 = b[0] + b[2]*Z3
dr_est = np.mean(mu1 - mu0 + T*(Y3-mu1)/p_hat - (1-T)*(Y3-mu0)/(1-p_hat))

print(f"interventional ORACLE ATE: {ate_oracle:+.3f}")
print(f"naive difference in means: {naive:+.3f}   (confounded — way off)")
print(f"standardization:           {std_est:+.3f}")
print(f"IPW:                       {ipw_est:+.3f}")
print(f"doubly robust:             {dr_est:+.3f}")

interventional ORACLE ATE: +1.502
naive difference in means: +3.341   (confounded — way off)
standardization:           +1.490
IPW:                       +1.498
doubly robust:             +1.498


**What just happened.** Four estimates against a measured oracle of **+1.502** (theory says exactly 1.5, since $E[1.5 - 0.5Z] = 1.5$ for $E[Z]=0$):

| estimator | estimate | error |
|---|---|---|
| naive difference in means | +3.341 | **+1.84** |
| standardization | +1.490 | −0.012 |
| IPW | +1.498 | −0.004 |
| doubly robust | +1.498 | −0.004 |

**The naive estimate is 2.2× the truth, and the mechanism is the same as Session 1 with a binary treatment.** High-$Z$ units are both more likely to be treated ($p = \sigma(1.5Z)$) *and* have higher $Y$ regardless of treatment ($+2.0Z$). So the treated group is systematically different from the untreated one before any treatment occurs, and the difference in means measures that gap plus the effect.

**Three routes fix it, and they are genuinely different routes — which is the point.** Standardization fits the outcome model and averages it over **everyone's** $Z$, simulating both worlds. IPW never models the outcome at all; it reweights each unit by $1/P(T \mid Z)$, upweighting units that got the treatment they were unlikely to get, **manufacturing the randomised trial nature refused to run**. Doubly robust combines them and is consistent if *either* model is correct.

**Do not read the tiny differences as a ranking.** The spread between 1.490 and 1.498 is 0.008, well inside Monte Carlo noise at $n = 40{,}000$ — rerun with another seed and the ordering shuffles. All three agree with the oracle; none is demonstrated better than the others here.

**And note what this demo structurally cannot show.** Both the outcome model (linear with a $T{\times}Z$ interaction, exactly matching the truth) and the propensity model (logistic in $Z$, exactly matching the truth) are **correctly specified**. Doubly robust is designed to survive when one of them is wrong — so a simulation where both are right cannot exhibit its advantage. **The estimator's headline property is untested here.** The experiment worth running: drop the interaction term from the outcome model, or fit the propensity without $Z^2$ when the truth needs it, and watch standardization or IPW break while DR holds.

**One practical hazard is visible in the propensity values.** At $Z = 3$, $p \approx 0.989$, so an untreated unit there carries weight $1/(1-p) \approx 90$. A handful of such units can dominate the IPW average. Here $Z$ is standard normal, the tails are thin, and it behaves — but with stronger confounding or heavier tails **IPW becomes extremely high-variance**, which is why weight trimming and overlap diagnostics are standard practice. Standardization has no analogous blow-up; it has the opposite weakness, relying entirely on the outcome model extrapolating correctly.

**Finally, the boundary that no estimator crosses.** All three assumed **no unmeasured confounding** — that $Z$ captures everything feeding both $T$ and $Y$. Add a hidden variable and every number above moves, while **the observed data looks identical**. No amount of cleverness detects it. The graph tells you what to adjust for; only the *design* tells you whether you measured enough — which is why a randomised trial outranks any estimator: it severs the arrow into $T$ by construction rather than by assumption.

**The honest boundary:** every method above assumed *no unmeasured confounding* — an assumption the data can never certify (that's what makes causal inference hard, and what instrumental variables, sensitivity analysis, and RCTs exist for). The graph tells you what to adjust; only design tells you whether you measured enough.

## 5. Conclusion

Seeing ≠ doing (naive slope 2× the truth, measured); backdoors are blocked by adjustment while colliders are *opened* by it (both manufactured on demand); and standardization/IPW/DR all recover the interventional oracle within noise. Regression answers questions about the world as it is; these tools answer questions about worlds we might make.

---
## Where next

- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) — the statistical machinery under each estimator.
- [Uncertainty in ML](./Uncertainty_in_ML.ipynb) — predictions under distribution shift: causality's sibling problem.